# Phase 2 — Serving + End-to-End Test
Loads Llama-3.1-8B-Instruct, runs the agent harness against real Tier 1/2/3 tasks, grades the results with grader.py.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi

## 1. Clone repo + install deps

In [ ]:
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune

In [ ]:
!pip install -q transformers accelerate bitsandbytes

## 2. Hugging Face login

In [ ]:
from huggingface_hub import login
login()

## 3. Sanity-check the parser (no GPU needed, run this first)

In [ ]:
!python envs/agent_harness.py

All 6 cases should say PASS before you go any further. If any FAIL, stop and fix `parse_tool_call` before burning GPU time.

## 4. Load the model

In [ ]:
import sys
sys.path.insert(0, ".")
from envs.agent_harness import load_model, run_agent
from envs.tools import TOOL_SCHEMAS, call_tool

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
print("Model loaded.")

## 5. Single-task smoke test before running the full suite

In [ ]:
tool_calls, final_text = run_agent(model, tok, "What's the weather in Tokyo?", TOOL_SCHEMAS, call_tool)
print("tool_calls:", tool_calls)
print("final_text:", final_text)

You should see one tool_call for get_weather with city=Tokyo, and final_text summarizing the result (since the harness feeds the tool result back and lets the model respond in a second turn).

## 6. Run all Tier 1 tasks end-to-end and grade them

In [ ]:
import json
from tasks.tier1 import TIER1_TASKS
from grader import grade_task

tier1_results = []
for task in TIER1_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool)
    grade = grade_task(task, tier=1, tool_calls=tool_calls, final_text=final_text)
    tier1_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

with open("results/tier1_baseline_results.json", "w") as f:
    json.dump(tier1_results, f, indent=2)

success_rate = sum(r["grade"]["success"] for r in tier1_results) / len(tier1_results)
print(f"\nTier 1 baseline success rate: {success_rate:.1%}")

## 7. (Optional) Run Tier 2 and Tier 3 the same way
Same pattern — swap the import and tier number. Tier 3 needs `final_text` for grading (already passed above), so no changes needed to the loop itself.

In [ ]:
from tasks.tier2 import TIER2_TASKS

tier2_results = []
for task in TIER2_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=2, tool_calls=tool_calls, final_text=final_text)
    tier2_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

with open("results/tier2_baseline_results.json", "w") as f:
    json.dump(tier2_results, f, indent=2)

success_rate = sum(r["grade"]["success"] for r in tier2_results) / len(tier2_results)
print(f"\nTier 2 baseline success rate: {success_rate:.1%}")

In [ ]:
from tasks.tier3 import TIER3_TASKS

tier3_results = []
for task in TIER3_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=3, tool_calls=tool_calls, final_text=final_text)
    tier3_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

with open("results/tier3_baseline_results.json", "w") as f:
    json.dump(tier3_results, f, indent=2)

success_rate = sum(r["grade"]["success"] for r in tier3_results) / len(tier3_results)
print(f"\nTier 3 baseline success rate: {success_rate:.1%}")

## 8. Download results to push from your laptop

In [ ]:
from google.colab import files
files.download("results/tier1_baseline_results.json")
files.download("results/tier2_baseline_results.json")
files.download("results/tier3_baseline_results.json")

Then locally:
```bash
# move the 3 downloaded files into results/
git add results/tier1_baseline_results.json results/tier2_baseline_results.json results/tier3_baseline_results.json
git commit -m "Phase 2: baseline (no optimization) results across all 3 tiers"
git push
```